In [1]:
from sage.all import *

# ------------------------------------------------------------
# Basic polynomial helpers
# ------------------------------------------------------------

def _is_zero_poly(p):
    return p == 0

def _deg(p):
    """
    Sage returns -1 for the degree of the zero polynomial in many cases,
    but this wrapper makes that behavior explicit.
    """
    if p == 0:
        return -1
    return p.degree()

def _lt(p):
    """
    Leading term with respect to ordinary degree.
    For univariate polynomials over Sage polynomial rings, p.lt() works.
    """
    if p == 0:
        return 0
    return p.lt()

def _normalize_weights(A, W=0, w0=1):
    """
    Row-operation version:
    W is indexed by columns, so len(W) = A.ncols().
    The paper's weight vector is (w0, w1, ..., wm).
    Your previous code implicitly used w0 = 1.
    """
    if W == 0:
        W = zero_vector(ZZ, A.ncols())
    return w0, vector(ZZ, W)


# ------------------------------------------------------------
# Row weighted degree and leading coordinate
# ------------------------------------------------------------

def WeightedDegreeOfRow(A, i, W=0, w0=1):
    """
    Row analogue of deg_w(A_j) in the paper.

    For row i:
        deg_w(A_i) = max_j { w0*deg(A[i,j]) + W[j] }.
    """
    w0, W = _normalize_weights(A, W, w0)

    if A[i].is_zero():
        return -Infinity

    return max(
        w0 * _deg(A[i, j]) + W[j]
        for j in range(A.ncols())
        if A[i, j] != 0
    )

def WeightedDegreeOfMatrix(A, W=0, w0=1):
    """
    Row analogue of deg_w(A) for the whole basis matrix.
    """
    w0, W = _normalize_weights(A, W, w0)

    degs = [
        WeightedDegreeOfRow(A, i, W, w0)
        for i in range(A.nrows())
        if not A[i].is_zero()
    ]

    if len(degs) == 0:
        return -Infinity

    return max(degs)

def LeadingCoordinateOfRow(A, i, W=0, w0=1):
    """
    Row analogue of LC_w(A_j).

    The paper uses the largest row index among critical positions.
    Since we are using rows, this returns the largest column index j such that

        w0*deg(A[i,j]) + W[j] = deg_w(row i).
    """
    w0, W = _normalize_weights(A, W, w0)

    if A[i].is_zero():
        return -1

    D = WeightedDegreeOfRow(A, i, W, w0)

    candidates = [
        j for j in range(A.ncols())
        if A[i, j] != 0 and w0 * _deg(A[i, j]) + W[j] == D
    ]

    if len(candidates) == 0:
        return -1

    return max(candidates)

def Pivot(A, i, W=0, w0=1):
    """
    Compatible with your earlier Pivot function.

    Returns:
        I  = leading coordinate / pivot column
        D  = ordinary degree of the pivot entry
        DW = weighted row degree
    """
    w0, W = _normalize_weights(A, W, w0)

    I = LeadingCoordinateOfRow(A, i, W, w0)

    if I == -1:
        return (-1, -1, -Infinity)

    D = _deg(A[i, I])
    DW = w0 * D + W[I]

    return (I, D, DW)

def is_reduced(A, W=0, w0=1):
    """
    Row analogue of reducedness.

    A is reduced if the nonzero rows have distinct leading coordinates.
    """
    w0, W = _normalize_weights(A, W, w0)

    nonzero_rows = [i for i in range(A.nrows()) if not A[i].is_zero()]
    leading_coords = [LeadingCoordinateOfRow(A, i, W, w0) for i in nonzero_rows]

    return len(set(leading_coords)) == len(leading_coords)


# ------------------------------------------------------------
# Accuracy-t approximation pi_t(A)
# ------------------------------------------------------------

def AccuracyApproximation(A, t, W=0, w0=1):
    """
    Row-operation analogue of Definition 3.9.

    Paper, column version:
        [pi_t(A)]_{i,j} = sum_{k : w0*k + w_i > deg_w(A_j) - t} a_k x^k.

    Row version:
        [pi_t(A)]_{i,j} = sum_{k : w0*k + W[j] > deg_w(row i) - t} a_k x^k.

    So the row degree controls the truncation, and W[j] is the weight
    of the column coordinate.
    """
    w0, W = _normalize_weights(A, W, w0)

    R = A.base_ring()
    x = R.gen()

    n, m = A.nrows(), A.ncols()
    A_pi = zero_matrix(R, n, m)

    for i in range(n):
        row_deg = WeightedDegreeOfRow(A, i, W, w0)

        if row_deg == -Infinity:
            continue

        for j in range(m):
            p = A[i, j]

            if p == 0:
                continue

            q = R(0)

            for k, ak in enumerate(p.list()):
                if ak != 0 and w0 * k + W[j] > row_deg - t:
                    q += ak * x**k

            A_pi[i, j] = q

    return A_pi


# ------------------------------------------------------------
# Algorithm 1, row-operation version
# ------------------------------------------------------------

def Algorithm1_Row(A, R, W=0, w0=1):
    """
    Row-operation analogue of Algorithm 1:
    Algorithm for computing U_w(A, 1).

    Paper column operation:
        A_{j1} <- A_{j1} + f A_{j2}
        U_{j1} <- U_{j1} + f U_{j2}

    Row analogue:
        A_{i1} <- A_{i1} + f A_{i2}
        U_{i1} <- U_{i1} + f U_{i2}

    Output U satisfies:
        U * A
    is either reduced or has smaller weighted degree.
    """
    w0, W = _normalize_weights(A, W, w0)

    A = copy(A)
    n = A.nrows()

    # Line 1: U <- I_m
    U = identity_matrix(R, n)

    Dold = Infinity
    Dnew = Infinity

    # Line 2: repeat
    while True:

        # Line 3: if A is reduced with respect to deg_w then
        if is_reduced(A, W, w0):

            # Line 4: return U
            return U

        # Line 5: else

        # Line 6: Find i1 != i2 such that j = LC_w(A_i1) = LC_w(A_i2)
        found = False
        i1 = i2 = j = None

        leading_coords = [
            LeadingCoordinateOfRow(A, i, W, w0)
            for i in range(n)
        ]

        for a in range(n):
            if A[a].is_zero():
                continue

            for b in range(a + 1, n):
                if A[b].is_zero():
                    continue

                if leading_coords[a] == leading_coords[b]:
                    i1, i2 = a, b
                    j = leading_coords[a]
                    found = True
                    break

            if found:
                break

        if not found:
            return U

        # Line 7: if deg(A[i1,j]) < deg(A[i2,j]) then
        if _deg(A[i1, j]) < _deg(A[i2, j]):

            # Line 8: Swap i1 and i2
            i1, i2 = i2, i1

        # Line 10:
        # f <- -LT(A[i1,j]) / LT(A[i2,j])
        #
        # In the paper this is ordinary leading term division.
        f = -_lt(A[i1, j]) / _lt(A[i2, j])

        # Make sure f is an element of the polynomial ring
        f = R(f)

        # Line 11: Dold <- deg_w(A_i1)
        Dold = WeightedDegreeOfRow(A, i1, W, w0)

        # Line 12: A_i1 <- A_i1 + f A_i2
        for col in range(A.ncols()):
            A[i1, col] = A[i1, col] + f * A[i2, col]

        # Line 13: U_i1 <- U_i1 + f U_i2
        for col in range(U.ncols()):
            U[i1, col] = U[i1, col] + f * U[i2, col]

        # Line 14: Dnew <- deg_w(A_i1)
        Dnew = WeightedDegreeOfRow(A, i1, W, w0)

        # Line 16: until Dnew < Dold
        if Dnew < Dold:

            # Line 17: return U
            return U


# ------------------------------------------------------------
# Algorithm 2, row-operation version, line by line
# ------------------------------------------------------------

def Alekhnovich(A, R, t, W=0, w0=1):
    """
    Row-operation analogue of Algorithm 2:
    The short-basis algorithm for computing U_w(A, t).

    Paper column version:
        A0 = A * U0
        return U0 * U(A0, remaining)

    Row version:
        A0 = U0 * A
        return U(A0, remaining) * U0

    Output U satisfies:
        U * A
    is the transformed basis.
    """
    w0, W = _normalize_weights(A, W, w0)

    if t < 1:
        raise ValueError("Algorithm 2 assumes t >= 1.")

    A = copy(A)

    # Line 1: A <- pi_t(A)
    A = AccuracyApproximation(A, t, W, w0)

    # Line 2: if t = 1 then
    if t == 1:

        # Line 3: return U(A, 1) // Use Algorithm 1
        return Algorithm1_Row(A, R, W, w0)

    # Line 4: else

    # Line 5: U0 <- U(A, floor(t/2)) // Recursive call
    U0 = Alekhnovich(A, R, t // 2, W, w0)

    # Line 6: A0 <- A * U0 in the paper.
    # Row-operation analogue:
    A0 = U0 * A

    # Line 7:
    # Paper:
    #   return U0 * U(A0, t - (deg_w(A) - deg_w(A0)))
    #
    # Row-operation analogue:
    #   return U(A0, remaining) * U0
    remaining = t - (
        WeightedDegreeOfMatrix(A, W, w0)
        - WeightedDegreeOfMatrix(A0, W, w0)
    )

    U1 = Alekhnovich(A0, R, remaining, W, w0)

    return U1 * U0

In [3]:
F = GF(5)
R = PolynomialRing(F, "x")
x = R.gen()

A = Matrix(R, [
    [x**2,     2*x,     1],
    [x**3 + 1, 3*x**2,  2*x**2],
    [1,        4*x,     3*x],
])

t = 3

U = Alekhnovich(A, R, t)
print("U =")
print(U)

print("U*A =")
print(U*A)

print("is reduced:", is_reduced(U*A))


KeyboardInterrupt

